# Finetuning Qwen3.5

> Originally, adapted from [Qwen3_5_(0_8B)_Vision.ipynb](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Qwen3_5_(0_8B)_Vision.ipynb#scrollTo=gGFzmplrEy9I)

![Qwen3.5](https://qianwen-res.oss-accelerate.aliyuncs.com/logo_qwen3.5.png)

Qwen3.5 features the following enhancement:

- **Unified Vision-Language Foundation**: Early fusion training on multimodal tokens achieves cross-generational parity with Qwen3 and outperforms Qwen3-VL models across reasoning, coding, agents, and visual understanding benchmarks.
- **Efficient Hybrid Architecture**: Gated Delta Networks combined with sparse Mixture-of-Experts deliver high-throughput inference with minimal latency and cost overhead.
- **Scalable RL Generalization**: Reinforcement learning scaled across million-agent environments with progressively complex task distributions for robust real-world adaptability.
- **Global Linguistic Coverage**: Expanded support to 201 languages and dialects, enabling inclusive, worldwide deployment with nuanced cultural and regional understanding.
- **Next-Generation Training Infrastructure**: Near-100% multimodal training efficiency compared to text-only training and asynchronous RL frameworks supporting massive-scale agent scaffolds and environment orchestration.

In [ ]:
import json
import random
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import display
from PIL import Image
from tqdm.auto import tqdm
from trl import SFTConfig, SFTTrainer
from unsloth import FastVisionModel
from unsloth.trainer import UnslothVisionDataCollator

from staged_qwen3_5_scivqa.config import (
    BASE_DIR,
    COMPETITION_DATA_DIR,
    ENABLE_THINKING,
    LORA_CHECKPOINT_YESNO,
    MODEL_ID,
    NUM_TRAIN_EPOCHS,
    PROMPT_YES_NO,
    TEMPERATURE,
    TOKEN_BUDGETS,
    TOP_K,
    TOP_P,
    MIN_P,
)
from staged_qwen3_5_scivqa.conversation import convert_to_conversation
from staged_qwen3_5_scivqa.context import get_paper_context
from staged_qwen3_5_scivqa.preprocessing import clean_answer
from staged_qwen3_5_scivqa.analysis import calculate_token_stats

In [ ]:
MAX_NEW_TOKENS = TOKEN_BUDGETS["Yes/No"]["max_new_tokens"]
MAX_SEQUENCE_LENGTH = TOKEN_BUDGETS["Yes/No"]["max_sequence_length"]

LORA_CHECKPOINT = LORA_CHECKPOINT_YESNO

CATEGORIES = ["train", "dev"]

In [ ]:
model, tokenizer = FastVisionModel.from_pretrained(
    MODEL_ID,
    load_in_4bit=False,  # Use 4bit to reduce memory use. False for 16bit LoRA.
    max_seq_length=MAX_SEQUENCE_LENGTH,  # Must match the max_length used during training
    use_gradient_checkpointing="unsloth",  # True or "unsloth" for long context
)

<a name="Data"></a>
### 🧪 Data Preparation

To convert our Sci-ImageMiner VQA data into the format required by Qwen2-VL (specifically for use with Unsloth), we need to restructure the data into a "messages" format.

The Qwen/Unsloth format expects a list of conversations where the image and the text prompt are bundled together in the user role, and the ground truth is in the assistant role, as follows:

```python
[
    { "role": "user",
    "content": [{"type": "text",  "text": Q}, {"type": "image", "image": image} ]
    },
    { "role": "assistant",
    "content": [{"type": "text",  "text": A} ]
    },
]
```

In [ ]:
PROMPT = PROMPT_YES_NO

In [ ]:
def load_dataset(category: str) -> list[dict]:
    case_dir = COMPETITION_DATA_DIR / category

    samples = []
    json_files = list(case_dir.rglob("*.json"))
    pbar = tqdm(json_files, desc="Processing Subfigures")

    # Simple trackers for your sanity
    valid_count = 0
    invalid_count = 0

    for json_file in pbar:
        fullpath = str(json_file)
        if (
            "content.json" in json_file.name
            or "images" not in fullpath
            or ".vscode" in fullpath
        ):
            continue

        pbar.set_description(f"Processing {json_file.name}")

        with open(json_file) as f:
            data = json.load(f)

        img_path = json_file.with_suffix(".jpg")
        assert img_path.exists(), f"{json_file.name} does not exist"

        # Open the full source image once
        full_img = Image.open(img_path.absolute())
        context = get_paper_context(json_file)

        # Extract bounding box info
        bboxes = data.get("bbox", {})

        # Iterate through subfigures (a, b, etc.) present in the VQA data
        for sub_key, q_list in data.get("vqa", {}).items():
            # Skip if there's no bounding box for this subfigure
            if sub_key not in bboxes:
                warnings.warn(f"Subfigure {sub_key} missing bbox in {json_file}")
                continue

            # Get coordinates and crop
            box = bboxes[sub_key]
            left = box["x"]
            top = box["y"]
            right = left + box["width"]
            bottom = top + box["height"]

            # Create the sub-image crop
            sub_image = full_img.crop((left, top, right, bottom))

            summary = data.get("summarization", {}).get(sub_key, None)
            table = data.get("data_extraction", {}).get(sub_key, None)

            # Process every question associated with this specific sub-figure
            for q_obj in q_list:
                question_text = q_obj.get("question") or q_obj.get("questions")
                question_type = q_obj.get("question_type", "")
                answer_type = q_obj.get("answer_type", "")

                if answer_type != "Yes/No":
                    continue

                human_prompt = PROMPT.format(
                    question=question_text,
                    question_type=question_type,
                    context=context,
                    summary=summary if summary is not None else "N/A",
                    table=table if table is not None else "N/A",
                )

                raw_response = q_obj.get("answer", "")

                # --- APPLY CLEANING HERE ---
                cleaned_response, is_valid = clean_answer(raw_response)

                # Skip invalid formats to keep the fine-tuning dataset pristine
                if not is_valid:
                    invalid_count += 1
                    continue

                valid_count += 1

                # Pass the cropped sub_image and the CLEANED response
                sample = convert_to_conversation(
                    human_prompt, sub_image, cleaned_response
                )
                samples.append(sample)

    return samples, valid_count, invalid_count

Let's convert the dataset into the "correct" format for finetuning:

In [ ]:
dataset = []
valid_count = 0
invalid_count = 0

for category in CATEGORIES:
    print(f"
Loading category: {category}")
    ds, vc, ic = load_dataset(category)

    dataset.extend(ds)
    valid_count += vc
    invalid_count += ic

print("-" * 40)
print("📋 DATASET CREATION SUMMARY")
print(f"Added to dataset (Valid): {valid_count}")
print(f"Skipped (Invalid format): {invalid_count}")
print("-" * 40)

We look at how the conversations are structured for the first example:

In [ ]:
dataset[0]["messages"][0]["content"][1]["image"]

In [ ]:
dataset[0]["messages"]

In [ ]:
# Run the analysis
df_tokens = calculate_token_stats(dataset, tokenizer)

# Display statistics
print("
--- Token Length Statistics ---")
display(
    df_tokens[["assistant_tokens", "total_tokens"]].describe(
        percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]
    )
)

# Plotting
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
df_tokens["assistant_tokens"].hist(
    bins=30, ax=axes[0], color="salmon", edgecolor="black"
)
axes[0].set_title("Assistant Tokens (for MAX_NEW_TOKENS)")
axes[0].set_xlabel("Token Count")

df_tokens["total_tokens"].hist(bins=30, ax=axes[1], color="skyblue", edgecolor="black")
axes[1].set_title("Total Sequence Tokens (for max_length)")
axes[1].set_xlabel("Token Count")
plt.show()

In [ ]:
# 1. Guard against silent truncation (e.g., if max_samples was used in stats)
if len(dataset) != len(df_tokens):
    raise ValueError(
        f"❌ Length mismatch: Dataset has {len(dataset)} samples, but token stats "
        f"have {len(df_tokens)} rows. Ensure you ran calculate_token_stats with max_samples=None."
    )

# 2. Guard against missing or corrupted data
if "total_tokens" not in df_tokens.columns:
    raise KeyError("❌ The DataFrame is missing the required 'total_tokens' column.")

if df_tokens["total_tokens"].isna().any():
    raise ValueError("❌ The 'total_tokens' column contains missing (NaN) values.")

# 3. Perform the actual filtering
original_size = len(dataset)

dataset = [
    sample
    for sample, total_tokens, assistant_tokens in zip(
        dataset, df_tokens["total_tokens"], df_tokens["assistant_tokens"], strict=False
    )
    if total_tokens <= MAX_SEQUENCE_LENGTH and assistant_tokens <= MAX_NEW_TOKENS
]

filtered_size = len(dataset)
dropped_count = original_size - filtered_size

print("-" * 40)
print("🧹 ROBUST TOKEN FILTERING SUMMARY")
print(f"Original size: {original_size}")
print(f"Max Length Allowed: {MAX_SEQUENCE_LENGTH}")
print(f"Kept (Safe): {filtered_size}")
print(f"Dropped (Outliers): {dropped_count}")
print("-" * 40)

In [ ]:
# Run the analysis again on the filtered dataset to confirm
df_tokens = calculate_token_stats(dataset, tokenizer)

# Display statistics
print("
--- Token Length Statistics ---")
display(
    df_tokens[["assistant_tokens", "total_tokens"]].describe(
        percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]
    )
)

# Plotting
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
df_tokens["assistant_tokens"].hist(
    bins=30, ax=axes[0], color="salmon", edgecolor="black"
)
axes[0].set_title("Assistant Tokens (for MAX_NEW_TOKENS)")
axes[0].set_xlabel("Token Count")

df_tokens["total_tokens"].hist(bins=30, ax=axes[1], color="skyblue", edgecolor="black")
axes[1].set_title("Total Sequence Tokens (for max_length)")
axes[1].set_xlabel("Token Count")
plt.show()

In [ ]:
print("
--- Class Imbalance Check & Upsampling ---")

yes_samples = []
no_samples = []

# 1. Separate the dataset into Yes and No categories
for sample in dataset:
    # Extract the assistant's response from the Unsloth/Qwen format
    answer = sample["messages"][1]["content"][0]["text"]
    if answer == "Yes":
        yes_samples.append(sample)
    elif answer == "No":
        no_samples.append(sample)

print(f"Original 'Yes' count: {len(yes_samples)}")
print(f"Original 'No' count: {len(no_samples)}")

# 2. Identify majority and minority classes
if len(yes_samples) > len(no_samples):
    majority_class, minority_class = yes_samples, no_samples
    minority_name = "No"
else:
    majority_class, minority_class = no_samples, yes_samples
    minority_name = "Yes"

imbalance_diff = len(majority_class) - len(minority_class)

# 3. Upsample if there is an imbalance
if imbalance_diff > 0:
    print(
        f"Upsampling minority class ('{minority_name}') by {imbalance_diff} samples..."
    )

    # Randomly sample from the minority class with replacement
    upsampled_additions = random.choices(minority_class, k=imbalance_diff)

    # Rebuild the dataset with balanced classes
    dataset = majority_class + minority_class + upsampled_additions

    # Shuffle to ensure the duplicated examples are distributed evenly
    random.seed(3407)  # using your notebook's random state
    random.shuffle(dataset)

    print(f"New dataset size after upsampling: {len(dataset)}")

    # Verification
    final_yes = sum(
        1 for s in dataset if s["messages"][1]["content"][0]["text"] == "Yes"
    )
    final_no = sum(1 for s in dataset if s["messages"][1]["content"][0]["text"] == "No")
    print(f"Final Count -> Yes: {final_yes} | No: {final_no}")
else:
    print("Classes are already perfectly balanced!")

<a name="Training"></a>
### 🚀 Training the model

Now let's train our model. We now add LoRA adapters for parameter efficient finetuning - this allows us to only efficiently train 1% of all parameters.

We do 60 steps to speed things up, but we can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`.

> We could also use `DPOTrainer` and `GRPOTrainer` for reinforcement learning!!

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,  # False if not finetuning vision layers
    finetune_language_layers=True,  # False if not finetuning language layers
    finetune_attention_modules=True,  # False if not finetuning attention layers
    finetune_mlp_modules=True,  # False if not finetuning MLP layers
    r=16,  # The larger, the higher the accuracy, but might overfit
    lora_alpha=16,  # Recommended alpha == r at least
    lora_dropout=0,
    bias="none",
    random_state=3407,
    use_rslora=False,  # We support rank stabilized LoRA
    loftq_config=None,  # And LoftQ
    # target_modules = "all-linear", # Optional now! Can specify a list if needed
)

In [ ]:
FastVisionModel.for_training(model)  # Enable for training!

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(
        model, tokenizer, max_seq_length=MAX_SEQUENCE_LENGTH, resize="max"
    ),  # https://github.com/unslothai/unsloth/issues/2764
    train_dataset=dataset,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_ratio=0.05,
        # max_steps=30,
        num_train_epochs=NUM_TRAIN_EPOCHS,  # Set this instead of max_steps for full training runs
        learning_rate=2e-4,
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.001,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",  # For Weights and Biases
        # You MUST put the below items for vision finetuning:
        remove_unused_columns=False,
        dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
        max_length=MAX_SEQUENCE_LENGTH,
    ),
)

In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
trainer_stats = trainer.train()

In [ ]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime'] / 60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

<a name="Saving"></a>
### 💾 Saving, loading finetuned models

To save the final model as LoRA adapters, either use Hugging Face's `push_to_hub` for an online save or `save_pretrained` for a local save.

> This **ONLY** saves the LoRA adapters, and not the full model.

In [ ]:
model.save_pretrained(LORA_CHECKPOINT)  # Local saving
tokenizer.save_pretrained(LORA_CHECKPOINT)
# model.push_to_hub(f"billsioros/{LORA_CHECKPOINT}", token = "YOUR_HF_TOKEN") # Online saving
# tokenizer.push_to_hub(f"billsioros/{LORA_CHECKPOINT}", token = "YOUR_HF_TOKEN") # Online saving